# Compass Analysis via SVD Archetype Space

This notebook implements a targeted approach to analysis of characters along a particular set of traits using a pre-computed Singular Value Decomposition (SVD) of character-trait data.

## Approach:
1. **Trait Selection**: Decide on "core" traits for your axis.
2. **Moral Vector ($A_m$)**: Create a vector in the trait space where only our chosen core traits are active.
3. **Archetype Space Projection**: Compute the chosen axis in archetype space using $U^T \times A_m$.
4. **Character Projection**: Project characters onto this axis to find characters most aligned / unaligned with your axis.
5. **Trait Alignment**: Find which other traits align with this axis.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set paths
data_path = 'data/plain/current/2000/'

def load_matrix(filename):
    return np.loadtxt(os.path.join(data_path, filename))

print("Loading data...")
traits_df = pd.read_csv(os.path.join(data_path, 'data_traits.tsv'), sep='\t')
characters_df = pd.read_csv(os.path.join(data_path, 'data_characters.tsv'), sep='\t')

U = load_matrix('data_archetype_space_U.txt')
A = load_matrix('data_raw_A.txt')
Sigma_vec = load_matrix('data_archetype_space_Sigma.txt')
V = load_matrix('data_archetype_space_V.txt')

# Ensure dimensions match U (464 traits)
k = U.shape[0]
Sigma_vec = Sigma_vec[:k]
V = V[:, :k]

Sigma = np.diag(Sigma_vec)

print(f"U shape: {U.shape}")
print(f"Sigma shape: {Sigma.shape}")
print(f"V shape: {V.shape}")
print(f"Traits: {len(traits_df)}, Characters: {len(characters_df)}")

Loading data...
U shape: (464, 464)
Sigma shape: (464,)
V shape: (2000, 464)
Traits: 464, Characters: 2000


## 1. Select Traits

You do this by making a config file with the traits you consider "core". See examples. Currently you only have to specify the trait you WANT out of the differential. This breaks for a few traits like "warm" (which is included in more than one differential. 

Currently, all traits are weighted equally.

Note: In this dataset, the $A$ matrix is typically constructed such that the `right pole` is positive and the `left pole` is negative. 
We will check the poles for each trait to ensure we assign the correct weight (e.g., +1 or -1) to represent the "Good" side.

In [2]:
# choose a config file to import traits from

# let's look at traits that make someone hobbity
# from config.trait_config_hobbit import core_traits

# a brief selection of traits in the spirit of POCS2
# from config.trait_config_pocs2 import core_traits

# or morality according to Anne, with more traits to test how that works
from config.trait_config_moral import core_traits

right_pole_matches = traits_df[traits_df['right pole'].isin(core_traits)][['index', 'differential']]
right_pole_matches['weight'] = -0.5

left_pole_matches = traits_df[traits_df['left pole'].isin(core_traits)][['index', 'differential']]
left_pole_matches['weight'] = 0.5

matches = pd.concat([right_pole_matches, left_pole_matches])
matches

,index,differential,weight
30,31,rude :: respectful,-0.5
83,84,cruel :: kind,-0.5
225,226,racist :: egalitarian,-0.5
314,315,stingy :: generous,-0.5
27,28,loyal :: traitorous,0.5
31,32,diligent :: lazy,0.5
170,171,open-minded :: close-minded,0.5


In [3]:
Am = np.zeros((U.shape[0], 1))
print(Am.shape)
for idx, data in matches.iterrows():
    # Subtract 1 because index in TSV is 1-based
    print(data['weight'])
    Am[idx] = data['weight']

print(f"Moral vector Am created with {matches.shape[0]} traits.")
pd.set_option('display.max_rows', 500) 

(464, 1)
-0.5
-0.5
-0.5
-0.5
0.5
0.5
0.5
Moral vector Am created with 7 traits.


## 2. Project into Archetype Space

The axis in the low-dimensional archetype space is given by $w_m = U^T A_m$.

In [4]:
wm = U.T @ Am
print(f"Moral axis wm shape: {wm.shape}")

Moral axis wm shape: (464, 1)


## 3. Project Characters onto the Moral Axis

Character coordinates in archetype space are $C = U^{T} A$.

In [5]:
C = U.T @ A
print("2000 characters in archetype space", C.shape)
print(C[:10])

2000 characters in archetype space (464, 2000)
[[ 2.1730315  3.1797615  2.7448767 ...  2.8141765  2.5683858 -0.8855492]
 [-1.8128114 -1.99624   -0.5641295 ... -1.2156008 -0.7076934 -1.6599769]
 [ 1.5417338 -0.4008947  1.133453  ... -1.6855709  0.0676395 -0.1626808]
 ...
 [-0.6789055 -0.4440058 -0.3430303 ...  0.6803284  0.4716941  0.1504852]
 [ 0.3433965  0.5602973  0.0473852 ...  0.2150412  0.0281909  0.2566883]
 [-0.2101988  0.0447063 -0.5851862 ...  0.3578359 -0.5716547  0.360949 ]]


In [6]:
# calculate euclidean distance between wm (trait axis in archetype space) and C (characters in archetype space)

P = np.zeros((1, C.shape[1]))

for i in range(C.shape[1]):
    distance = np.linalg.norm(C[:,i] - wm)
    P[0, i] = distance

P = P.flatten()

# get closest indices
indices = np.argpartition(P,10)[:10]
print(indices)

[ 413 1755 1265 1303 1268  254  373  547  375  664]


In [7]:
characters = characters_df.drop(columns='index')  # drop confusingly-indexed index column
pd.set_option('display.max_colwidth', None)  # show full card url
characters.iloc[indices][['character', 'card url']]

# this is generally giving characters who are closest to trait matrix [0,0,0,0,0,0,0]
# not really effective

,character,card url
413,Telemachus,https://compstorylab.org/archetypometrics/cards/The-Odyssey-Telemachus-2000-464-341.pdf
1755,Lambert,https://compstorylab.org/archetypometrics/cards/Alien-Lambert-2000-464-341.pdf
1265,Cyclops,https://compstorylab.org/archetypometrics/cards/X-Men-Cyclops-2000-464-341.pdf
1303,Mary Svevo,https://compstorylab.org/archetypometrics/cards/Eternal-Sunshine-of-the-Spotless-Mind-Mary-Svevo-2000-464-341.pdf
1268,Iceman,https://compstorylab.org/archetypometrics/cards/X-Men-Iceman-2000-464-341.pdf
254,Benvolio,https://compstorylab.org/archetypometrics/cards/Romeo-and-Juliet-Benvolio-2000-464-341.pdf
373,Michael 'Newmie' Newman,https://compstorylab.org/archetypometrics/cards/Baywatch-Michael-Newmie-Newman-2000-464-341.pdf
547,Colin Khoo,https://compstorylab.org/archetypometrics/cards/Crazy-Rich-Asians-Colin-Khoo-2000-464-341.pdf
375,Garner Ellerbee,https://compstorylab.org/archetypometrics/cards/Baywatch-Garner-Ellerbee-2000-464-341.pdf
664,Vincent Vega,https://compstorylab.org/archetypometrics/cards/Pulp-Fiction-Vincent-Vega-2000-464-341.pdf


In [8]:
# try cosine distance instead

P = np.zeros((1, C.shape[1]))

for i in range(C.shape[1]):
    distance = C[:,i] @ wm[:,0]
    P[0, i] = distance

P = P.flatten()

# get closest indices
indices = np.argpartition(P,10)[:10]
print(indices)

characters.iloc[indices][['character', 'card url']]

[1140 1708  399 1699  480 1523 1511 1036 1665    8]


,character,card url
1140,Alphonse Elric,https://compstorylab.org/archetypometrics/cards/Fullmetal-Alchemist-Brotherhood-Alphonse-Elric-2000-464-341.pdf
1708,Sam Obisanya,https://compstorylab.org/archetypometrics/cards/Ted-Lasso-Sam-Obisanya-2000-464-341.pdf
399,Penelope Garcia,https://compstorylab.org/archetypometrics/cards/Criminal-Minds-Penelope-Garcia-2000-464-341.pdf
1699,Ted Lasso,https://compstorylab.org/archetypometrics/cards/Ted-Lasso-Ted-Lasso-2000-464-341.pdf
480,Belle,https://compstorylab.org/archetypometrics/cards/Beauty-and-the-Beast-Belle-2000-464-341.pdf
1523,Anna Bates,https://compstorylab.org/archetypometrics/cards/Downton-Abbey-Anna-Bates-2000-464-341.pdf
1511,Kimmy Schmidt,https://compstorylab.org/archetypometrics/cards/Unbreakable-Kimmy-Schmidt-Kimmy-Schmidt-2000-464-341.pdf
1036,Forrest Gump,https://compstorylab.org/archetypometrics/cards/Forrest-Gump-Forrest-Gump-2000-464-341.pdf
1665,Dale Cooper,https://compstorylab.org/archetypometrics/cards/Twin-Peaks-Dale-Cooper-2000-464-341.pdf
8,Abby Sciuto,https://compstorylab.org/archetypometrics/cards/NCIS-Abby-Sciuto-2000-464-341.pdf
